# 01 — Fetch and Cache Raw Data

Fetches match data from StatsBomb open data and understat.com, caches to `data/raw/` as JSON.
Idempotent — re-running skips already-fetched files.

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from src.statsbomb_loader import (
    get_available_seasons, get_matches, get_starters,
    get_substitutions, get_shots, get_player_names
)

seasons = get_available_seasons()
print(seasons)

In [ ]:
for _, season in seasons.iterrows():
    season_id = int(season['season_id'])
    season_name = season['season_name']
    out_file = RAW_DIR / f'sb_matches_{season_id}.json'
    if out_file.exists():
        print(f'[SKIP] {season_name}')
        continue

    print(f'[FETCH] {season_name} ...')
    matches = get_matches(season_id)
    records = []

    for _, match in matches.iterrows():
        mid = int(match['match_id'])
        home = str(match['home_team'])
        away = str(match['away_team'])
        try:
            starters = get_starters(mid)
            subs = get_substitutions(mid, home, away)
            shots = get_shots(mid)
            names = get_player_names(mid)
        except Exception as e:
            print(f'  [ERROR] match {mid}: {e}')
            continue

        records.append({
            'match_id': f'sb_{mid}',
            'date': str(match['match_date']),
            'home_team': home,
            'away_team': away,
            'home_team_id': home,
            'away_team_id': away,
            'season': season_name,
            'source': 'statsbomb',
            'starters': starters,
            'substitutions': subs,
            'shots': shots,
            'player_names': names
        })

    with open(out_file, 'w') as f:
        json.dump(records, f)
    print(f'  Saved {len(records)} matches to {out_file.name}')

In [ ]:
from src.understat_loader import (
    get_season_results, get_match_shots, get_match_roster,
    parse_starters, parse_substitutions, parse_shots
)

# 5 seasons back from 2025-26: starting years 2020, 2021, 2022, 2023, 2024
UNDERSTAT_YEARS = [2020, 2021, 2022, 2023, 2024]

for year in UNDERSTAT_YEARS:
    season_label = f'{year}-{str(year + 1)[-2:]}'
    out_file = RAW_DIR / f'us_matches_{year}.json'
    if out_file.exists():
        print(f'[SKIP] {season_label}')
        continue

    print(f'[FETCH] {season_label} ...')
    results = get_season_results(year)
    records = []

    for match in results:
        mid = int(match['id'])
        home_team = match['h']['title']
        away_team = match['a']['title']
        home_id = f"us_team_{match['h']['id']}"
        away_id = f"us_team_{match['a']['id']}"
        date_str = match['datetime'][:10]

        try:
            shots_raw = get_match_shots(mid)
            roster_raw = get_match_roster(mid)
        except Exception as e:
            print(f'  [ERROR] match {mid}: {e}')
            continue

        home_starters = parse_starters(roster_raw, 'h')
        away_starters = parse_starters(roster_raw, 'a')
        home_subs = parse_substitutions(roster_raw, 'h', home_id)
        away_subs = parse_substitutions(roster_raw, 'a', away_id)
        home_shots = parse_shots(shots_raw, 'h', home_id, mid)
        away_shots = parse_shots(shots_raw, 'a', away_id, mid)

        player_names = {
            f"us_{p['id']}": p['player']
            for p in roster_raw.get('h', []) + roster_raw.get('a', [])
        }

        records.append({
            'match_id': f'us_{mid}',
            'date': date_str,
            'home_team': home_team,
            'away_team': away_team,
            'home_team_id': home_id,
            'away_team_id': away_id,
            'season': season_label,
            'source': 'understat',
            'starters': {home_team: home_starters, away_team: away_starters},
            'substitutions': home_subs + away_subs,
            'shots': home_shots + away_shots,
            'player_names': player_names
        })

    with open(out_file, 'w') as f:
        json.dump(records, f)
    print(f'  Saved {len(records)} matches to {out_file.name}')

print('Done!')